# Q5+6.
## 5.
```{admonition}
:class: note
In Chapter 4, we used logistic regression to predict the probability of `default` using `income` and `balance` on the `Default` data set. We will now estimate the test error of this logistic regression model using the validation set approach.

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
import statsmodels.api as sm

In [2]:
default_df = pd.read_csv("../../../ALL CSV FILES - 2nd Edition/Default.csv")
default_df['default'] = default_df['default'] == 'Yes'

### (a)
```{admonition}
:class: note
Fit a logistic regression model that uses `income` and `balance` to predict `default`.

In [3]:
X = default_df[['income','balance']]
Y = default_df['default']

lr_all = LogisticRegression(fit_intercept=True)
lr_all.fit(X,Y.values)
print(lr_all.coef_[0])
print(lr_all.intercept_[0])

[2.08089741e-05 5.64710265e-03]
-11.540467918644863


In [4]:
lr_all_sm = sm.GLM(Y,sm.add_constant(X),family=sm.families.Binomial())
results = lr_all_sm.fit()

### (b)
```{admonition}
:class: note
Using the validation set approach, estimate the test error of this model.

In [5]:
X_train, X_test, Y_train, Y_test = train_test_split(sm.add_constant(X),Y,random_state=1728)
lr_sm_val = sm.GLM(Y_train,X_train,family=sm.families.Binomial())
results = lr_sm_val.fit()
predictions = results.predict(X_test) > 0.5
(predictions != Y_test).mean()

np.float64(0.0324)

In [6]:
from sklearn.metrics import accuracy_score

lr = LogisticRegression()
lr.fit(X_train,Y_train)
1-accuracy_score(lr.predict(X_test), Y_test)

0.032399999999999984

### (c)
```{admonition}
:class: note
Repeat the process in (b) three times, using three different splits of the observations into a training set and a validation set.

In [7]:
for i in range(3):
    X_train, X_test, Y_train, Y_test = train_test_split(sm.add_constant(X),Y,random_state=1728+i)
    lr_sm_val = sm.GLM(Y_train,X_train,family=sm.families.Binomial())
    results = lr_sm_val.fit()
    predictions = results.predict(X_test) > 0.5
    print((predictions != Y_test).mean())

0.0324
0.0256
0.0276


### (d)
```{admonition}
:class: note
Now consider a logistic regression model that predicts the probability of `default` using `income`, `balance`, and a dummy variable for `student`. Estimate the test error for this model using the validation set approach.

In [8]:
X['student'] = (default_df['student'] == 'Yes').astype(int)

X_train, X_test, Y_train, Y_test = train_test_split(sm.add_constant(X),Y,random_state=1728)
lr_sm_val = sm.GLM(Y_train,X_train,family=sm.families.Binomial())
results = lr_sm_val.fit()
predictions = results.predict(X_test) > 0.5
print((predictions != Y_test).mean())

0.0324


## 6.
```{admonition}
:class: note
We continue to consider the use of a logistic regression model to predict the probability of `default` using `income` and `balance` on the `Default` data set. In particular, we will now compute estimates for the standard errors of the `income` and `balance` logistic regression coefficients in two different ways: (1) using the bootstrap, and (2) using the standard formula for computing the standard errors in the `sm.GLM()` function.

### (a)
```{admonition}
:class: note
Using the `summarize()` and `sm.GLM()` functions, determine the estimated standard errors for the coefficients associated with `income` and `balance` in a multiple logistic regression model that uses both predictors.

In [9]:
results = lr_all_sm.fit()
results.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                 Generalized Linear Model Regression Results                  
==============================================================================
Dep. Variable:                default   No. Observations:                10000
Model:                            GLM   Df Residuals:                     9997
Model Family:                Binomial   Df Model:                            2
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -789.48
Date:                Tue, 28 Apr 2026   Deviance:                       1579.0
Time:                        21:33:26   Pearson chi2:                 6.95e+03
No. Iterations:                     9   Pseudo R-squ. (CS):             0.1256
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const        -11.5405      0.435    -26.544      0.000     -12.393     -10.688
income      2.081e-05   4.99e-06      4.174      0.000     1.1e-05    3.06e-05
balance        0.0056      0.000     24.835      0.000       0.005       0.006
==============================================================================
"""

### (b)
```{admonition}
:class: note
Write a function, `boot_fn()`, that takes as input the `Default` data set as well as an index of the observations, and that outputs the coefficient estimates for `income` and `balance` in the multiple logistic regression model.

In [10]:
def boot_fn(df: pd.DataFrame, n: np.array) -> tuple:
    D = df.iloc[n]
    X = D[['income','balance']]
    Y = D[['default']]
    model = sm.GLM(Y,sm.add_constant(X),family=sm.families.Binomial())
    return model.fit().params    

### (c)
```{admonition}
:class: note
Use your `boot_fn()` function to estimate the standard errors of the logistic regression coefficients for `income` and `balance`.

In [11]:
rng = np.random.default_rng(1728)

In [12]:
B = 1000
n = default_df.shape[0]
parameters = {'intercept': np.empty(B), 'income':np.empty(B), 'balance':np.empty(B)}
for i in range(B):
    parameters['intercept'][i], parameters['income'][i], parameters['balance'][i] = boot_fn(default_df,rng.choice(n,n,replace=True))

In [13]:
std_estimates = np.empty(3)
for i,p in enumerate(parameters):
    std_estimates[i] = np.std(parameters[p],ddof=1)
    print('Estimate for standard deviation of {} is {}'.format(p,std_estimates[i]))

Estimate for standard deviation of intercept is 0.4349310771261158
Estimate for standard deviation of income is 4.9006453764849395e-06
Estimate for standard deviation of balance is 0.0002277644119395577


### (d)
```{admonition}
:class: note
Comment on the estimated standard errors obtained using the `sm.GLM()` function and using the bootstrap.

In [19]:
np.isclose(results.bse.values, std_estimates,0.05)

array([ True,  True,  True])

They are approximately the same.